In [1]:
import numpy as np
import pandas as pd 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv('real_estate_data.csv').drop(22)

In [3]:
df.shape

(246, 7)

In [4]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Salon', 'Restaurant', 'Spa'..."
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Bowling Alley', 'Mini Theatre', 'Manicured G..."
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,"['Terrace Garden', 'Gazebo', 'Fountain', 'Amph..."
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Volley Ball Court', 'Aerobi..."
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,"['Mini Theatre', 'Doctor on Call', 'Concierge ..."


In [5]:
df[["PropertyName","TopFacilities"]][0:10]

,PropertyName,TopFacilities
0,Smartworld One DXP,"['Swimming Pool', 'Salon', 'Restaurant', 'Spa'..."
1,M3M Crown,"['Bowling Alley', 'Mini Theatre', 'Manicured G..."
2,Adani Brahma Samsara Vilasa,"['Terrace Garden', 'Gazebo', 'Fountain', 'Amph..."
3,Sobha City,"['Swimming Pool', 'Volley Ball Court', 'Aerobi..."
4,Signature Global City 93,"['Mini Theatre', 'Doctor on Call', 'Concierge ..."
5,Whiteland The Aspen,"['Reflexology Park', 'Card Room', 'High Speed ..."
6,Bestech Altura,"['Swimming Pool', 'Football', 'Flower Garden',..."
7,Elan The Presidential,"['Swimming Pool', 'Aerobics Centre', 'Shopping..."
8,Signature Global City 92,"['Swimming Pool', 'Reflexology Park', 'Terrace..."
9,Emaar Digihomes,"['Mini Theatre', 'Swimming Pool', 'Salon', 'Ca..."


In [6]:

import re


def extract_list(s):
    return re.findall(r"'(.*?)'", s)

df['TopFacilities'] = df['TopFacilities'].apply(extract_list)

In [7]:
df.head(1)

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","[Swimming Pool, Salon, Restaurant, Spa, Cafete..."


In [8]:
df['Facilities']=df['TopFacilities'].apply(' '.join)

In [9]:

tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

In [10]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Facilities'])

In [11]:
tfidf_matrix.toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.14379378,
        0.14379378],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(246, 953))

In [12]:
cosine_sim1 = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [13]:
cosine_sim1.shape

(246, 246)

In [14]:
def rec_prop(property_name , cosine_sim = cosine_sim1):
    idx = df.index[df['PropertyName'] == property_name].tolist()[0]
    sim_score = list(enumerate(cosine_sim[idx]))
    sim_score = sorted(sim_score, key=lambda x: x[1], reverse=True)
    sim_score = sim_score[1:6]
    pidx = [i[0] for i in sim_score]
    recomendation_df =  pd.DataFrame({
        'PropertyName': df['PropertyName'].iloc[pidx].values,
        'similarity_score': [i[1] for i in sim_score]
    })
    return recomendation_df

In [15]:
idx = df.index[df['PropertyName'] == 'M3M Crown'].tolist()[0]

In [16]:
rec_prop('M3M Crown')

,PropertyName,similarity_score
0,DLF The Ultima,0.358603
1,BPTP Pedestal,0.331763
2,Ireo Victory Valley,0.318586
3,M3M Sky Lofts,0.309176
4,Central Park Flower Valley Mikasa Plots,0.297431


In [17]:
sim_score = list(enumerate(cosine_sim1[idx]))

In [18]:
sim_score

[(0, np.float64(0.010951589538864515)),
 (1, np.float64(1.0000000000000002)),
 (2, np.float64(0.019821211060493036)),
 (3, np.float64(0.1296332817539559)),
 (4, np.float64(0.15888474057768504)),
 (5, np.float64(0.019310272324387357)),
 (6, np.float64(0.21700920287542233)),
 (7, np.float64(0.0084010073622641)),
 (8, np.float64(0.041269095672560804)),
 (9, np.float64(0.07992243485499474)),
 (10, np.float64(0.013031464375486135)),
 (11, np.float64(0.03223439184656369)),
 (12, np.float64(0.14896009351131334)),
 (13, np.float64(0.11517012009261494)),
 (14, np.float64(0.029979471390467638)),
 (15, np.float64(0.011399439203418632)),
 (16, np.float64(0.0)),
 (17, np.float64(0.01918649830583175)),
 (18, np.float64(0.06587679948437511)),
 (19, np.float64(0.03734590222631334)),
 (20, np.float64(0.21621658180837194)),
 (21, np.float64(0.0452474461444466)),
 (22, np.float64(0.05640567615225693)),
 (23, np.float64(0.15373791911523915)),
 (24, np.float64(0.09988347503176563)),
 (25, np.float64(0.1731

In [19]:
sim_score = sorted(sim_score, key=lambda x: x[1], reverse=True)

In [20]:
sim_score =sim_score[1:6]

In [21]:
pidx = [i[0] for i in sim_score]
pidx

[85, 226, 74, 145, 89]

In [22]:
pd.DataFrame({
    'PropertyName': df['PropertyName'].iloc[pidx],
    'similarity_score': sim_score
})

,PropertyName,similarity_score
86,DLF The Ultima,"(85, 0.35860320390686057)"
227,BPTP Pedestal,"(226, 0.33176299517252417)"
75,Ireo Victory Valley,"(74, 0.31858623396699676)"
146,M3M Sky Lofts,"(145, 0.3091761490494428)"
90,Central Park Flower Valley Mikasa Plots,"(89, 0.29743060271126887)"


In [23]:
df.head(1)

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities,Facilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","[Swimming Pool, Salon, Restaurant, Spa, Cafete...",Swimming Pool Salon Restaurant Spa Cafeteria S...


In [24]:
df[['PropertyName', 'PriceDetails']]['PriceDetails'][0]

"{'2 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '1,370 sq.ft.', 'price-range': '₹ 2 - 2.4 Cr'}, '3 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '1,850 - 2,050 sq.ft.', 'price-range': '₹ 2.25 - 3.59 Cr'}, '4 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '2,600 sq.ft.', 'price-range': '₹ 3.24 - 4.56 Cr'}}"

In [25]:
import ast
import re
import numpy as np

# area parser

In [26]:
def parse_area(area):

    nums = re.findall(r"\d+(?:,\d+)?(?:\.\d+)?", str(area))

    nums = [float(x.replace(",", "")) for x in nums]

    if len(nums)==1:
        return nums[0], nums[0]

    elif len(nums)>=2:
        return min(nums), max(nums)

    return np.nan, np.nan

# price parser

In [27]:
def parse_price(price):

    nums = re.findall(r"\d+(?:\.\d+)?", str(price))

    vals=[]

    for n in nums:

        vals.append(float(n))

    if "L" in price:

        vals=[x/100 for x in vals]

    if len(vals)==1:

        return vals[0],vals[0]

    elif len(vals)>=2:

        return min(vals),max(vals)

    return np.nan,np.nan

In [28]:
def extract_features(detail):

    try:
        detail = ast.literal_eval(detail)
    except:
        return {}

    output={}

    for bhk,info in detail.items():

        output[f"{bhk}_building_type"]=info.get("building_type")

        low,high=parse_area(info.get("area"))

        output[f"{bhk}_area_low"]=low
        output[f"{bhk}_area_high"]=high

        low,high=parse_price(info.get("price-range"))

        output[f"{bhk}_price_low"]=low
        output[f"{bhk}_price_high"]=high
        output[f"{bhk}_area_type"] = info.get("area_type")

    return output

In [29]:
feature_df = df["PriceDetails"].apply(extract_features)

feature_df = pd.json_normalize(feature_df)

df_final = pd.concat(
    [
        df['PropertyName'],
        feature_df
    ],
    axis=1
)

In [30]:
df_final.set_index("PropertyName",inplace=True)

In [31]:
df_final.iloc[0]

2 BHK_building_type      Apartment
2 BHK_area_low              1370.0
2 BHK_area_high             1370.0
2 BHK_price_low                2.0
2 BHK_price_high               2.4
2 BHK_area_type        Carpet Area
3 BHK_building_type      Apartment
3 BHK_area_low              1850.0
3 BHK_area_high             2050.0
3 BHK_price_low               2.25
3 BHK_price_high              3.59
3 BHK_area_type        Carpet Area
4 BHK_building_type      Apartment
4 BHK_area_low              2600.0
4 BHK_area_high             2600.0
4 BHK_price_low               3.24
4 BHK_price_high              4.56
4 BHK_area_type        Carpet Area
Land_building_type             NaN
Land_area_low                  NaN
Land_area_high                 NaN
Land_price_low                 NaN
Land_price_high                NaN
Land_area_type                 NaN
5 BHK_building_type            NaN
5 BHK_area_low                 NaN
5 BHK_area_high                NaN
5 BHK_price_low                NaN
5 BHK_price_high    

In [32]:
cat_col = df_final.select_dtypes(include='object').columns.tolist()

C:\Users\HP\AppData\Local\Temp\ipykernel_25848\2276582199.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_col = df_final.select_dtypes(include='object').columns.tolist()


In [33]:
cat_col

['2 BHK_building_type',
 '2 BHK_area_type',
 '3 BHK_building_type',
 '3 BHK_area_type',
 '4 BHK_building_type',
 '4 BHK_area_type',
 'Land_building_type',
 'Land_area_type',
 '5 BHK_building_type',
 '5 BHK_area_type',
 '1 BHK_building_type',
 '1 BHK_area_type',
 '1 RK_building_type',
 '1 RK_area_type',
 '6 BHK_building_type',
 '6 BHK_area_type']

In [34]:
ohe_df = pd.get_dummies(df_final,columns=cat_col, drop_first=True, dtype="uint8")

In [35]:
ohe_df.fillna(0,inplace=True)

,2 BHK_area_low,2 BHK_area_high,2 BHK_price_low,2 BHK_price_high,3 BHK_area_low,3 BHK_area_high,3 BHK_price_low,3 BHK_price_high,4 BHK_area_low,4 BHK_area_high,...,4 BHK_area_type_Super Built-up Area,5 BHK_building_type_Independent Floor,5 BHK_building_type_Villa,5 BHK_area_type_Carpet Area,5 BHK_area_type_Super Built-up Area,1 BHK_building_type_Service Apartment,1 BHK_area_type_Super Built-up Area,1 RK_area_type_Super Built-up Area,6 BHK_building_type_Villa,6 BHK_area_type_Super Built-up Area
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,1370.0,1370.0,2.0000,2.4000,1850.0,2050.0,2.25,3.59,2600.0,2600.0,...,0,0,0,0,0,0,0,0,0,0
M3M Crown,0.0,0.0,0.0000,0.0000,1605.0,2170.0,2.20,3.03,2248.0,2670.0,...,1,0,0,0,0,0,0,0,0,0
Adani Brahma Samsara Vilasa,0.0,0.0,0.0000,0.0000,1800.0,3150.0,2.43,15.75,2750.0,4500.0,...,1,0,0,0,0,0,0,0,0,0
Sobha City,1381.0,1692.0,1.5500,3.2100,1711.0,2343.0,1.76,4.79,2423.0,2963.0,...,1,0,0,0,0,0,0,0,0,0
Signature Global City 93,981.0,1118.0,0.0106,0.9301,1235.0,1530.0,1.12,1.45,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,964.0,964.0,1.3500,1.3500,1127.0,1127.0,1.55,1.55,1562.0,1562.0,...,1,0,0,0,0,0,0,0,0,0
Pyramid Urban Homes 2,500.0,625.0,0.0000,0.0000,0.0,0.0,0.00,0.00,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
Satya The Hermitage,1450.0,1450.0,0.8000,0.8000,1991.0,1991.0,1.00,1.00,2639.0,4711.0,...,1,0,0,0,1,0,0,0,0,0


In [36]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

norm_ohe_df = pd.DataFrame(scaler.fit_transform(ohe_df),columns=ohe_df.columns,index=ohe_df.index)

In [37]:
norm_ohe_df

,2 BHK_area_low,2 BHK_area_high,2 BHK_price_low,2 BHK_price_high,3 BHK_area_low,3 BHK_area_high,3 BHK_price_low,3 BHK_price_high,4 BHK_area_low,4 BHK_area_high,...,4 BHK_area_type_Super Built-up Area,5 BHK_building_type_Independent Floor,5 BHK_building_type_Villa,5 BHK_area_type_Carpet Area,5 BHK_area_type_Super Built-up Area,1 BHK_building_type_Service Apartment,1 BHK_area_type_Super Built-up Area,1 RK_area_type_Super Built-up Area,6 BHK_building_type_Villa,6 BHK_area_type_Super Built-up Area
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,1.223499,1.020101,1.229266,1.063789,0.553787,0.370864,0.490560,0.414174,0.602838,0.212073,...,-0.848693,-0.111111,-0.216353,-0.171139,-0.400471,-0.111111,-0.194871,-0.063888,-0.063888,-0.111111
M3M Crown,-0.893541,-0.896660,-0.418097,-0.446627,0.293086,0.472749,0.462219,0.245937,0.382746,0.243172,...,1.178282,-0.111111,-0.216353,-0.171139,-0.400471,-0.111111,-0.194871,-0.063888,-0.063888,-0.111111
Adani Brahma Samsara Vilasa,-0.893541,-0.896660,-0.418097,-0.446627,0.500583,1.304803,0.592590,4.067311,0.696627,1.056178,...,1.178282,-0.111111,-0.216353,-0.171139,-0.400471,-0.111111,-0.194871,-0.063888,-0.063888,-0.111111
Sobha City,1.240497,1.470610,0.858610,1.573554,0.405879,0.619632,0.212813,0.774681,0.492166,0.373342,...,1.178282,-0.111111,-0.216353,-0.171139,-0.400471,-0.111111,-0.194871,-0.063888,-0.063888,-0.111111
Signature Global City 93,0.622383,0.667529,-0.409366,0.138722,-0.100626,-0.070634,-0.149960,-0.228730,-1.022842,-0.943016,...,-0.848693,-0.111111,-0.216353,-0.171139,-0.400471,-0.111111,-0.194871,-0.063888,-0.063888,-0.111111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,0.596113,0.452068,0.693873,0.402982,-0.215547,-0.412795,0.093778,-0.198688,-0.046184,-0.249074,...,1.178282,-0.111111,-0.216353,-0.171139,-0.400471,-0.111111,-0.194871,-0.063888,-0.063888,-0.111111
Pyramid Urban Homes 2,-0.120899,-0.022226,-0.418097,-0.446627,-1.414772,-1.369658,-0.784811,-0.664343,-1.022842,-0.943016,...,-0.848693,-0.111111,-0.216353,-0.171139,-0.400471,-0.111111,-0.194871,-0.063888,-0.063888,-0.111111
Satya The Hermitage,1.347122,1.132029,0.240848,0.056845,0.703823,0.320771,-0.217979,-0.363920,0.627223,1.149918,...,1.178282,-0.111111,-0.216353,-0.171139,2.497057,-0.111111,-0.194871,-0.063888,-0.063888,-0.111111


In [38]:
cosine_sim2 = cosine_similarity(norm_ohe_df,norm_ohe_df)

In [39]:
cosine_sim2

array([[ 1.        , -0.28748044, -0.12094965, ..., -0.1195517 ,
        -0.20151881, -0.09260065],
       [-0.28748044,  1.        ,  0.13281758, ..., -0.07037406,
        -0.06747354, -0.09791125],
       [-0.12094965,  0.13281758,  1.        , ..., -0.13878337,
        -0.32277409, -0.27623977],
       ...,
       [-0.1195517 , -0.07037406, -0.13878337, ...,  1.        ,
         0.10459656,  0.18266425],
       [-0.20151881, -0.06747354, -0.32277409, ...,  0.10459656,
         1.        ,  0.94301388],
       [-0.09260065, -0.09791125, -0.27623977, ...,  0.18266425,
         0.94301388,  1.        ]], shape=(246, 246))

In [40]:
def recommend_properties_with_scores(property_name, top_n=5):
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim2[norm_ohe_df.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = norm_ohe_df.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    
    return recommendations_df

# Test the recommender function using a property name
recommend_properties_with_scores('M3M Golf Hills')

,PropertyName,SimilarityScore
0,BPTP Terra,0.968899
1,AIPL The Peaceful Homes,0.968865
2,Unitech Escape,0.968816
3,DLF Regal Gardens,0.950464
4,Sobha City,0.948994


# 3 recommendation sys

In [41]:
def parse_location(x):

    if pd.isna(x):
        return {}

    if isinstance(x, dict):
        return x

    try:
        return ast.literal_eval(x)
    except:
        return {}

In [42]:
def distance_to_km(distance):

    if pd.isna(distance):
        return np.nan

    distance = str(distance).strip().lower()

    num = re.findall(r"\d+\.?\d*", distance)

    if len(num)==0:
        return np.nan

    num = float(num[0])

    if "m" in distance and "km" not in distance:
        return num/1000

    return num

In [43]:
def normalize_location_name(name):

    name = name.lower().strip()

    replacements = {

        "nh48":"nh-48",
        "nh 48":"nh-48",
        "nh-48":"nh-48",

        "igi airport":"airport",
        "indira gandhi international airport":"airport",
        "airport":"airport",

        "rapid metro":"metro",
        "metro station":"metro",
        "metro":"metro",

        "railway station":"railway station",

        "hospital":"hospital",

        "school":"school"

    }

    return replacements.get(name, name)

In [44]:
rows = []

for loc in df["LocationAdvantages"]:

    loc = parse_location(loc)

    temp = {}

    for place, dist in loc.items():

        place = normalize_location_name(place)

        temp[place] = distance_to_km(dist)

    rows.append(temp)

location_df = pd.DataFrame(rows, index=df["PropertyName"])

In [45]:
location_df1 = location_df.fillna(location_df.max().max() + 5)

In [46]:
location_df1

,bajghera road,palam vihar halt,dpsg palam vihar,park hospital,gurgaon railway station,the northcap university,dwarka expy,hyatt place gurgaon udyog vihar,"dwarka sector 21, metro station",pacific d21 mall,...,mcc cricket ground dhankot,the shri ram school aravali,taj city centre gurugram,minda industries corporate office,"rampura flyover, naurangpur rd",manesar toll plaza - kherki daula,"imt manesar, gurugram",holiday inn,sector 84 road,skyview corporate park
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,0.8,2.5,3.1,3.1,4.9,5.4,1.2,7.7,7.2,7.4,...,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5
M3M Crown,59.5,59.5,59.5,59.5,59.5,4.4,59.5,59.5,59.5,8.2,...,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5
Adani Brahma Samsara Vilasa,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,...,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5
Sobha City,59.5,59.5,59.5,59.5,59.5,9.2,59.5,59.5,59.5,59.5,...,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5
Signature Global City 93,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,...,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,...,59.5,3.9,4.0,59.5,59.5,59.5,59.5,59.5,59.5,59.5
Pyramid Urban Homes 2,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,...,59.5,59.5,59.5,3.1,3.7,6.1,7.7,3.0,59.5,59.5
Satya The Hermitage,59.5,59.5,59.5,59.5,3.3,8.8,59.5,59.5,59.5,59.5,...,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5,59.5


In [54]:
import pickle
with open("location_df.pkl",'wb') as f:
    pickle.dump(location_df1,f)

In [47]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

location_scaled = pd.DataFrame(scaler.fit_transform(location_df1),columns=location_df1.columns,index=location_df1.index)

In [48]:

location_scaled

,bajghera road,palam vihar halt,dpsg palam vihar,park hospital,gurgaon railway station,the northcap university,dwarka expy,hyatt place gurgaon udyog vihar,"dwarka sector 21, metro station",pacific d21 mall,...,mcc cricket ground dhankot,the shri ram school aravali,taj city centre gurugram,minda industries corporate office,"rampura flyover, naurangpur rd",manesar toll plaza - kherki daula,"imt manesar, gurugram",holiday inn,sector 84 road,skyview corporate park
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,0.004241,0.0,0.0,0.039182,0.056995,0.049209,0.020003,0.119796,0.0,0.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
M3M Crown,1.000000,1.0,1.0,1.000000,1.000000,0.031634,1.000000,1.000000,1.0,0.015355,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Adani Brahma Samsara Vilasa,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Sobha City,1.000000,1.0,1.0,1.000000,1.000000,0.115993,1.000000,1.000000,1.0,1.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Signature Global City 93,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,...,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Pyramid Urban Homes 2,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
Satya The Hermitage,1.000000,1.0,1.0,1.000000,0.029361,0.108963,1.000000,1.000000,1.0,1.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [49]:
cosine_sim3 = cosine_similarity(location_scaled)

In [50]:
cosine_sim3.shape

(246, 246)

In [61]:
def recommend_properties_with_scores(property_name, top_n=247):
    
    cosine_sim_matrix = 30*cosine_sim1 + 20*cosine_sim2 + 8*cosine_sim3
    # cosine_sim_matrix = cosine_sim3
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim_matrix[location_df1.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = location_df1.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    
    return recommendations_df

# Test the recommender function using a property name
recommend_properties_with_scores('Ireo Victory Valley')

,PropertyName,SimilarityScore
0,DLF The Crest,35.481459
1,Pioneer Urban Presidia,35.353374
2,Ambience Creacions,34.663120
3,SS The Leaf,31.661376
4,Silverglades The Melia,31.290804
...,...,...
240,JMS The Nation,-6.107699
241,Vatika Aspiration,-6.164759
242,JMS Prime Land,-6.342487
243,Corona Greens,-6.368617


In [190]:
(3*cosine_sim3 + 5*cosine_sim2 + 6*cosine_sim1).shape

(246, 246)

In [58]:
with open("cosine_sim1.pkl",'wb') as f:
    pickle.dump(cosine_sim1,f)

In [59]:
with open("cosine_sim2.pkl",'wb') as f:
    pickle.dump(cosine_sim2,f)

In [60]:
with open("cosine_sim3.pkl",'wb') as f:
    pickle.dump(cosine_sim3,f)